### RAG pipeline - Data Ingestion to Vector DB pipeline 

In [26]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [27]:
# Read all the pdf's in the directory
def process_all_pdf(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.append(doc)
            print(f" ✓ Loaded: {len(documents)} pages")
        except Exception as e:
            print(f" ✗ Error: {e}")
    
    print(f" Total documents loaded: {len(all_documents)}")
    return all_documents

# Process all documents in the directory 
all_pdf_documents = process_all_pdf("../data")

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JBFKNL+AdvTT99c4c969', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(527, 0, 6110133328), '/LastChar': 116, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 531, 531, 531, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 635, 0, 583, 677, 510, 500, 0, 666, 0, 385, 572, 489, 822, 666, 697, 552, 0, 0, 510, 520, 656, 0, 0, 0, 0, 562, 0, 0, 0, 0, 0, 0, 500, 0, 447, 0, 510, 0, 0, 562, 250, 0, 500, 0, 0, 562, 0, 583, 0, 354, 416, 343]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JBFLGJ+AdvTT577c760c', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(537, 0, 6110133328), '/LastChar': 121, '/Subtype': '/Type1', '/Type': '/Font

Found 2 PDF files to process

Processing: machine learning and deep learning.pdf


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JBFMJK+AdvTT50a2f13e.I', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(226, 0, 6110133328), '/LastChar': 122, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [0, 0, 0, 0, 0, 0, 770, 0, 322, 322, 0, 0, 250, 322, 250, 270, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 322, 0, 0, 0, 0, 0, 0, 604, 604, 666, 718, 604, 604, 718, 718, 322, 437, 0, 552, 822, 666, 718, 604, 718, 604, 500, 552, 718, 604, 822, 604, 0, 0, 385, 0, 385, 0, 0, 0, 500, 500, 437, 500, 437, 270, 500, 500, 270, 270, 437, 270, 718, 500, 500, 500, 0, 385, 385, 270, 500, 437, 666, 437, 437, 385]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/JBFLLJ+AdvTTc488b0e6', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor

 ✓ Loaded: 11 pages

Processing: attention is all you need.pdf
 ✓ Loaded: 11 pages
 Total documents loaded: 2


In [28]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Arbortext Advanced Print Publisher 9.1.440/W Unicode', 'creationdate': '2021-04-08T13:34:38+08:00', 'author': 'Christian Janiesch', 'keywords': 'Machine learning,Deep learning,Artificial intelligence,Artificial neural networks,Analytical model building,C6,C8,M15,O3', 'moddate': '2021-04-08T13:35:16+08:00', 'subject': 'Electron Markets, doi:10.1007/s12525-021-00475-2', 'title': 'Machine learning and deep learning', 'source': '../data/pdf/machine learning and deep learning.pdf', 'total_pages': 11, 'page': 10, 'page_label': '11', 'source_file': 'machine learning and deep learning.pdf', 'file_type': 'pdf'}, page_content='play. Science, 362 (6419), 1140 –1144. https://doi.org/10.1126/\nscience.aar6404.\nSpooner, T., Fearnley, J., Savani, R., & Koukorinis, A. (2018). Market\nmaking via reinfor cement learning. Proceedings of the 17th\nInternational Conference on Autonomous Agents and MultiAgent\nsystems,4 3 4–4

In [29]:
# Text splitting get into chunk 

def split_documents(documents):
    ''' splitting documents into smaller chunks for better RAG performance'''
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000,
        chunk_overlap = 200,
        length_function = len,
        separators=["\n\n", "\n"," ", ""]

    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk: ")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs


In [30]:
chunks = split_documents(all_pdf_documents)
chunks

Split 2 documents into 7 chunks

Example chunk: 
Content: play. Science, 362 (6419), 1140 –1144. https://doi.org/10.1126/
science.aar6404.
Spooner, T., Fearnley, J., Savani, R., & Koukorinis, A. (2018). Market
making via reinfor cement learning. Proceedings 
Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Arbortext Advanced Print Publisher 9.1.440/W Unicode', 'creationdate': '2021-04-08T13:34:38+08:00', 'author': 'Christian Janiesch', 'keywords': 'Machine learning,Deep learning,Artificial intelligence,Artificial neural networks,Analytical model building,C6,C8,M15,O3', 'moddate': '2021-04-08T13:35:16+08:00', 'subject': 'Electron Markets, doi:10.1007/s12525-021-00475-2', 'title': 'Machine learning and deep learning', 'source': '../data/pdf/machine learning and deep learning.pdf', 'total_pages': 11, 'page': 10, 'page_label': '11', 'source_file': 'machine learning and deep learning.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Arbortext Advanced Print Publisher 9.1.440/W Unicode', 'creationdate': '2021-04-08T13:34:38+08:00', 'author': 'Christian Janiesch', 'keywords': 'Machine learning,Deep learning,Artificial intelligence,Artificial neural networks,Analytical model building,C6,C8,M15,O3', 'moddate': '2021-04-08T13:35:16+08:00', 'subject': 'Electron Markets, doi:10.1007/s12525-021-00475-2', 'title': 'Machine learning and deep learning', 'source': '../data/pdf/machine learning and deep learning.pdf', 'total_pages': 11, 'page': 10, 'page_label': '11', 'source_file': 'machine learning and deep learning.pdf', 'file_type': 'pdf'}, page_content='play. Science, 362 (6419), 1140 –1144. https://doi.org/10.1126/\nscience.aar6404.\nSpooner, T., Fearnley, J., Savani, R., & Koukorinis, A. (2018). Market\nmaking via reinfor cement learning. Proceedings of the 17th\nInternational Conference on Autonomous Agents and MultiAgent\nsystems,4 3 4–4

### Embedding and vector store DB

In [31]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [36]:
class EmbeddingManager:
    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):

        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name} ")
            self.model = SentenceTransformer(self.model_name)
            print(f" Model loaded successfully . Embedding dimensions : {self.model.get_sentence_embedding_dimension}")
        except Exception as e:
            print(f"Error loading model : {self.model_name} : {e}")
            raise

    def generate_embeddings(self, texts: list[str]) -> np.ndarray :

        if not self.model:
            raise ValueError("Model not loaded ")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7191.71it/s]


 Model loaded successfully . Embedding dimensions : <bound method SentenceTransformer.get_sentence_embedding_dimension of SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)>


### Vector Store 

In [ ]:
class VectorStore:

    def __init__ (self, collection_name: str = "pdf_documents", persist_directory: str = " data/vector_store"):
        
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initalize_store(self):
        